# Lesson 9.4 — Modernization: Serverless, Containers & Purpose-Built Databases

**Level: Advanced** | **Exam:** SAP-C02 Domain 4, Task 4.4; Domain 3, Tasks 3.3 & 3.4

## Learning Objectives
- Apply the Strangler Fig pattern for incremental modernization
- Design a target architecture replacing EC2 + RDS with serverless and containers
- Select purpose-built databases to replace generic relational databases
- Identify decoupling opportunities using SQS, SNS, and Step Functions

## Modernization Strategies

### Strangler Fig Pattern
Replace a monolith piece by piece:
```
1. API Gateway in front of existing monolith
2. Route new features → Lambda/containers
3. Gradually migrate existing routes
4. Eventually decommission the monolith
```

### Database Modernization Opportunities
| From | To | Benefit |
|------|-----|--------|
| MySQL on EC2 | Aurora MySQL | Managed, auto-storage, 5x throughput |
| Oracle | Aurora PostgreSQL | Eliminate Oracle licensing ($$$) |
| MySQL RDS | DynamoDB | Unlimited scale, single-ms reads |
| Homegrown search | OpenSearch | Full-text, analytics |

### Containerization Path
```
Phase 1: Dockerize → ECR → ECS/EKS (lift and shift containers)
Phase 2: Split into microservices
Phase 3: Service mesh + observability (X-Ray, App Mesh)
```

### Decouple for Scale
```
Before: Monolith handles everything synchronously
After:
  API GW → Lambda (sync ops)
  SQS Queue → Lambda (async ops)
  EventBridge → Fan-out to multiple processors
  Step Functions → Multi-step workflows
```

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Modernization opportunity analysis
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')
rds = boto3.client('rds', region_name='us-east-1')

instances = []
paginator = ec2.get_paginator('describe_instances')
for page in paginator.paginate(Filters=[{'Name': 'instance-state-name', 'Values': ['running']}]):
    for reservation in page['Reservations']:
        for inst in reservation['Instances']:
            name = next((t['Value'] for t in inst.get('Tags', []) if t['Key'] == 'Name'), 'Unknown')
            instances.append({'id': inst['InstanceId'], 'type': inst['InstanceType'], 'name': name})

if instances:
    print('Running EC2 Instances (modernization candidates):')
    for inst in instances:
        if inst['type'].startswith('t3.micro'):
            path = 'Consider Lambda (if stateless + event-driven)'
        elif inst['type'].startswith('t3'):
            path = 'Consider ECS Fargate (containerize)'
        else:
            path = 'Assess: containers or right-size'
        print(f"  {inst['id']:20} {inst['type']:15} {inst['name']:30}")
        print(f"    Modernization: {path}")
else:
    print('No running EC2 found.')

# Modernization priority matrix
print('\n=== Modernization Priority Matrix ===')
apps = [
    ('Stateless REST API', 'Low', 'High', 'Refactor to Lambda + API Gateway'),
    ('Batch ETL jobs', 'Low', 'High', 'Refactor to Lambda + Step Functions'),
    ('MySQL on EC2 (app DB)', 'Low', 'High', 'Replatform to Aurora MySQL'),
    ('Stateful session app', 'Medium', 'Medium', 'Replatform to ECS + ElastiCache'),
    ('Oracle (core ERP)', 'Very High', 'Low', 'Retain now; plan refactor in 18 months'),
]
print(f"{'Application':35} {'Complexity':12} {'Value':8} Recommendation")
for name, complexity, value, rec in apps:
    print(f"  {name:33} {complexity:12} {value:8} {rec}")

## Exercise

In [ ]:
# TODO: Design the target modernized architecture for this monolith:
# Current: Single EC2 (Node.js Express), MySQL RDS, Memcached on EC2
# Endpoints:
#   GET /products (read-heavy — 90% of traffic, cacheable)
#   POST /orders (ACID required, write-once)
#   POST /emails/send (slow, non-blocking acceptable)
#   GET /search?q= (full-text search)

modernized = {
    'GET /products': {
        'compute': 'TODO',       # Lambda / ECS / API Gateway
        'database': 'TODO',      # DynamoDB / Aurora / ElastiCache
        'caching': 'TODO',       # CloudFront / ElastiCache / None
        'reason': 'TODO'
    },
    'POST /orders': {
        'compute': 'TODO',
        'database': 'TODO',      # Must be ACID
        'pattern': 'TODO',       # Sync / Queue / Event
        'reason': 'TODO'
    },
    'POST /emails/send': {
        'compute': 'TODO',
        'pattern': 'TODO',       # Should this be async?
        'service': 'TODO',       # SES / SQS + Lambda?
        'reason': 'TODO'
    },
    'GET /search': {
        'compute': 'TODO',
        'database': 'TODO',      # Full-text search service?
        'reason': 'TODO'
    }
}

import json
print(json.dumps(modernized, indent=2))

## Summary
- **Strangler Fig**: Incremental modernization via API Gateway routing — lower risk than big-bang rewrite
- **Decouple async**: Convert slow synchronous operations to SQS + Lambda → better resilience
- **Database modernization**: Biggest ROI — eliminate Oracle licensing, move to Aurora/DynamoDB
- **Containerization**: Docker → ECR → ECS/EKS; decompose monolith gradually into services
- **Observe first**: Use X-Ray + CloudWatch to find bottlenecks before modernizing

---
**Final Phase:** [Capstone Project — Production-Grade Multi-Account AWS Environment](../capstone/capstone-project.ipynb)